---

## PIIMiddleware 옵션별 테스트

``PIIMiddleware`` 는 대화에서 PII(개인 식별 정보)를 감지하고 처리합니다.

- 훅: ``before_model`` (입력·도구 결과), ``after_model`` (AI 출력)

**참고:** [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

§1~§9는 LLM 없이 ``sanitize_human_input`` / ``sanitize_ai_output`` 으로 처리 결과만 검증합니다.
§10~§12는 ``MiddlewarePIIAgent`` 로 에이전트 전체를 실행합니다 (`OPENAI_API_KEY` 필요).

**옵션 요약:**

| 옵션 | 예시 | 역할 |
|:---|:---|:---|
| ``pii_type`` | ``"email"``, ``"credit_card"``, ``"ip"``, ``"mac_address"``, ``"url"``, ``"api_key"`` | 감지 대상 (내장 5종 + 커스텀 이름) |
| ``strategy`` | ``"redact"``, ``"mask"``, ``"hash"``, ``"block"`` | 감지 시 처리 방식 |
| ``detector`` | ``r"sk-[a-zA-Z0-9]{32}"`` | 커스텀 정규식 또는 감지 함수 |
| ``apply_to_input`` | ``True`` (기본) | 사용자 ``HumanMessage`` 검사 |
| ``apply_to_output`` | ``False`` (기본) | ``AIMessage`` 검사 |
| ``apply_to_tool_results`` | ``False`` (기본) | ``ToolMessage`` 검사 |

**``strategy``**

| 값 | 결과 예시 |
|:---|:---|
| ``redact`` | ``[REDACTED_EMAIL]`` |
| ``mask`` | ``************1111`` (신용카드) |
| ``hash`` | ``Reach me at <email_hash:fb98d44a>`` (매칭 구간만) |
| ``block`` | ``PIIDetectionError`` 예외 |

**내장 ``pii_type`` (5종)**

| 타입 | 감지 대상 | § |
|:---|:---|:---|
| ``email`` | 이메일 | 1 |
| ``credit_card`` | 신용카드 (Luhn 검증) | 2 |
| ``ip`` | IPv4/IPv6 | 4 |
| ``mac_address`` | MAC 주소 | 5 |
| ``url`` | HTTP(S)·bare URL | 6 |
| *(커스텀)* | ``detector`` 정규식 등 | 3 (api_key 예시) |

In [1]:
from langchain_core.messages import HumanMessage

from feature.MiddlewarePII import (
    MiddlewarePIIAgent,
    PIIDetectionError,
    make_default_pii_middlewares,
    make_pii_middleware,
    sanitize_ai_output,
    sanitize_human_input,
)
from feature.MiddlewarePII import _DEFAULT_API_KEY_DETECTOR

### 1. ``email`` + ``strategy="redact"`` — 이메일 수정

사용자 입력에서 이메일을 ``[REDACTED_EMAIL]`` 로 치환합니다.

In [2]:
email_mw = make_pii_middleware("email", strategy="redact", apply_to_input=True)

raw = "My email is teddy@example.com. Can you help?"
sanitized = sanitize_human_input(email_mw, raw)

assert "[REDACTED_EMAIL]" in sanitized
assert "teddy@example.com" not in sanitized
print("✓ email redact —", sanitized)

✓ email redact — My email is [REDACTED_EMAIL]. Can you help?


### 2. ``credit_card`` + ``strategy="mask"`` — 신용카드 마스킹

Luhn 검증을 통과하는 번호만 감지됩니다. (예: ``4111111111111111``)

In [3]:
cc_mw = make_pii_middleware("credit_card", strategy="mask", apply_to_input=True)

raw = "My credit card is 4111111111111111"
sanitized = sanitize_human_input(cc_mw, raw)

assert "4111111111111111" not in sanitized
assert "1111" in sanitized  # 마지막 4자리 유지
print("✓ credit_card mask —", sanitized)

✓ credit_card mask — My credit card is ************1111


### 3. 커스텀 ``api_key`` + 정규식 ``detector`` — API 키 마스킹
- 출력값의 대한 포맷 변경은 Middleware 자체를 상속받아 커스텀 해야됨

In [4]:
api_mw = make_pii_middleware(
    "api_key", # custom name
    detector=_DEFAULT_API_KEY_DETECTOR, # 정규식 or funtion
    strategy="mask",
    apply_to_input=True,
)

raw = "My API key is sk-12345678901234567890123456789012"
sanitized = sanitize_human_input(api_mw, raw)

assert "sk-12345678901234567890123456789012" not in sanitized
print("✓ api_key mask —", sanitized)

✓ api_key mask — My API key is ****9012


### 4. ``ip`` + ``strategy="redact"`` — IP 주소 수정

IPv4 주소를 ``[REDACTED_IP]`` 로 치환합니다.

In [5]:
ip_mw = make_pii_middleware("ip", strategy="redact", apply_to_input=True)

raw = "Server at 192.168.1.100 is down"
sanitized = sanitize_human_input(ip_mw, raw)

assert "[REDACTED_IP]" in sanitized
assert "192.168.1.100" not in sanitized
print("✓ ip redact —", sanitized)

✓ ip redact — Server at [REDACTED_IP] is down


### 5. ``mac_address`` + ``strategy="mask"`` — MAC 주소 마스킹

MAC 주소의 앞부분을 마스킹하고 마지막 옥텟만 남깁니다.

In [6]:
mac_mw = make_pii_middleware("mac_address", strategy="mask", apply_to_input=True)

raw = "Device MAC is AA:BB:CC:DD:EE:FF"
sanitized = sanitize_human_input(mac_mw, raw)

assert "AA:BB:CC" not in sanitized
assert "FF" in sanitized
print("✓ mac_address mask —", sanitized)

✓ mac_address mask — Device MAC is **:**:**:**:**:FF


### 6. ``url`` — ``redact`` / ``mask``

``http``/``https`` URL 과 bare URL 모두 감지합니다.

- strategy가 ``redact``, ``mask`` 모두 출력값이 유사하지만 버그는 일단 아님

In [8]:
raw_url = "Visit https://secret.example.com/path for details"

url_redact = sanitize_human_input(
    make_pii_middleware("url", strategy="redact", apply_to_input=True),
    raw_url,
)
assert "[REDACTED_URL]" in url_redact
assert "secret.example.com" not in url_redact
print("✓ url redact —", url_redact)

url_mask = sanitize_human_input(
    make_pii_middleware("url", strategy="mask", apply_to_input=True),
    raw_url,
)
assert "[MASKED_URL]" in url_mask
assert "secret.example.com" not in url_mask
print("✓ url mask —", url_mask)

✓ url redact — Visit [REDACTED_URL] for details
✓ url mask — Visit [MASKED_URL] for details


### 7. ``strategy="block"`` — ``PIIDetectionError``

PII 감지 시 예외를 발생시켜 실행을 막습니다.

In [9]:
block_mw = make_pii_middleware("email", strategy="block", apply_to_input=True)

try:
    sanitize_human_input(block_mw, "Contact me at user@example.com")
    raise AssertionError("email block — PIIDetectionError 기대")
except PIIDetectionError as e:
    print(f"✓ strategy=block — {type(e).__name__}: {e}")

✓ strategy=block — PIIDetectionError: Detected 1 instance(s) of email in text content


### 8. ``strategy="hash"`` — 결정적 해시 치환

감지된 PII **부분만** ``<email_hash:fb98d44a>`` 형태로 바꿉니다 (문장 앞부분은 유지).

In [12]:
hash_mw = make_pii_middleware("email", strategy="hash", apply_to_input=True)

sanitized = sanitize_human_input(hash_mw, "Reach me at a@b.com")

assert "<email_hash:" in sanitized  # 매칭된 이메일 부분만 해시로 치환
assert "a@b.com" not in sanitized
print("✓ strategy=hash —", sanitized)

✓ strategy=hash — Reach me at <email_hash:fb98d44a>


### 9. ``apply_to_output=True`` — AI 출력 검사

In [13]:
output_mw = make_pii_middleware(
    "email",
    strategy="redact",
    apply_to_input=False,
    apply_to_output=True,
)

sanitized = sanitize_ai_output(output_mw, "Send mail to secret@corp.com")

assert "[REDACTED_EMAIL]" in sanitized
print("✓ apply_to_output —", sanitized)

✓ apply_to_output — Send mail to [REDACTED_EMAIL]


### 10. ``make_default_pii_middlewares()`` — 기본 3종 조합

원본 노트북: email ``redact`` + credit_card ``mask`` + api_key ``mask``.

In [14]:
default_mws = make_default_pii_middlewares()
print("미들웨어:", [m.name for m in default_mws])
assert len(default_mws) == 3

미들웨어: ['PIIMiddleware[email]', 'PIIMiddleware[credit_card]', 'PIIMiddleware[api_key]']


### 11. ``MiddlewarePIIAgent`` — 에이전트 전체 실행

PII가 포함된 입력을내도 에이전트가 응답합니다 (입력은 미들웨어에서 정화됨).

In [15]:
agent = MiddlewarePIIAgent()

pii_message = (
    "My credit card is 4111111111111111, "
    "API key sk-12345678901234567890123456789012, "
    "email teddy@example.com. What's the weather in Seoul?"
)

result = agent.invoke(
    inputs={"messages": [HumanMessage(content=pii_message)]},
)

print("✓ 에이전트 응답:", result["messages"][-1].content[:300])


🔄 Node: PIIMiddleware[email].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

My credit card is 4111111111111111, API key sk-12345678901234567890123456789012, email [REDACTED_EMAIL]. What's the weather in Seoul?

🔄 Node: PIIMiddleware[credit_card].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

My credit card is ************1111, API key sk-12345678901234567890123456789012, email [REDACTED_EMAIL]. What's the weather in Seoul?

🔄 Node: PIIMiddleware[api_key].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

My credit card is ************1111, API key ****9012, email [REDACTED_EMAIL]. What's the weather in Seoul?

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
===============

### 12. ``stream()`` — 스트리밍 실행

In [19]:
agent_stream = MiddlewarePIIAgent()

agent_stream.stream(
    inputs={"messages": [HumanMessage(content="Weather in Tokyo?")]},
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
It's sunny in Tokyo!
🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
The weather in Tokyo is sunny!